# NFL Play-Calling GRPO Training 🏈

**Train an LLM to be an NFL Offensive Coordinator using GRPO (Group Relative Policy Optimization)**

The model learns to call plays (formation + play type) given a defensive formation, with rewards sampled from real NFL play outcomes (16,000+ plays from the 2022 NFL season).

**Stack**: Unsloth + TRL GRPOTrainer + Qwen2.5-1.5B-Instruct + LoRA

**Runtime**: Select **T4 GPU** or better (Runtime → Change runtime type → GPU)

In [ ]:
# Cell 1: Install dependencies (pin vllm to TRL-compatible version)
!pip install -q "vllm==0.10.2"
!pip install -q unsloth trl datasets matplotlib
!pip install -q --upgrade transformers

In [ ]:
# Cell 2: Clone repo and load lookup table
!git clone https://github.com/ad-fletcher/football-play-caller.git repo 2>/dev/null || echo "Already cloned"

import json
import os
import random
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import torch
from datasets import Dataset
from transformers import TrainerCallback
from trl import GRPOConfig, GRPOTrainer

with open("repo/data/lookup_table.json") as f:
    _data = json.load(f)

LOOKUP = _data["lookup"]
DEFENSE_NAMES = list(_data["defense_distribution"].keys())
DEFENSE_WEIGHTS = list(_data["defense_distribution"].values())

print(f"Loaded lookup table: {len(LOOKUP)} entries")
print(f"Defensive formations: {DEFENSE_NAMES}")

VALID_FORMATIONS = ["SHOTGUN", "SINGLEBACK", "EMPTY", "I_FORM", "PISTOL"]
VALID_PLAY_TYPES = ["run", "pass", "play_action"]

## Model & LoRA Setup

Load Qwen2.5-1.5B-Instruct in 4-bit with LoRA rank 32. The base model is frozen — only the small LoRA adapter weights (~5M params) are trained.

In [ ]:
# Cell 3: Load model with Unsloth
from unsloth import FastLanguageModel

MODEL_NAME = "unsloth/Qwen2.5-1.5B-Instruct"
MAX_SEQ_LENGTH = 512
LORA_RANK = 32

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    fast_inference=True,
    max_lora_rank=LORA_RANK,
    gpu_memory_utilization=0.5,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    lora_alpha=LORA_RANK,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

print(f"Model loaded: {MODEL_NAME}")
print(f"Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

## Prompt & Dataset

Each training sample gives the model a game situation with a randomly sampled defensive formation. The model must output a JSON play call.

In [ ]:
# Cell 4: Prompts and dataset

SYSTEM_PROMPT = """\
You are an NFL offensive coordinator. Given the game situation and defensive formation, call a play.

Respond with ONLY a JSON object in this exact format:
{"formation": "<FORMATION>", "play_type": "<PLAY_TYPE>"}

Valid formations: SHOTGUN, SINGLEBACK, EMPTY, I_FORM, PISTOL
Valid play types: run, pass, play_action"""


def make_prompt(defense: str) -> list[dict]:
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"1st & 10 from your own 25. Defense: {defense}. Call your play."},
    ]


NUM_TRAIN_PROMPTS = 4096
random.seed(42)
defenses = random.choices(DEFENSE_NAMES, weights=DEFENSE_WEIGHTS, k=NUM_TRAIN_PROMPTS)
dataset = Dataset.from_dict({
    "prompt": [make_prompt(d) for d in defenses],
    "defense_formation": defenses,
})

print(f"Dataset: {len(dataset)} prompts")
print(f"Sample prompt:\n{json.dumps(dataset[0]['prompt'], indent=2)}")

## Reward Functions

Two reward signals:
1. **Format reward** — Did the model output valid JSON with valid formation/play_type? (+1.0 valid, -5.0 invalid)
2. **Play reward** — Yards gained sampled from real NFL data for that (formation, play_type, defense) combo

In [ ]:
# Cell 5: Reward functions

def _extract_text(completion) -> str:
    """Get raw text from a completion (handles both chat and string format)."""
    if isinstance(completion, list):
        return completion[0]["content"]
    return str(completion)


def _parse_action(text: str):
    """Parse JSON action from model output. Returns (formation, play_type) or None."""
    text = text.strip()
    start = text.find("{")
    end = text.rfind("}") + 1
    if start == -1 or end == 0:
        return None
    try:
        parsed = json.loads(text[start:end])
        formation = parsed.get("formation", "")
        play_type = parsed.get("play_type", "")
        if formation in VALID_FORMATIONS and play_type in VALID_PLAY_TYPES:
            return (formation, play_type)
    except (json.JSONDecodeError, TypeError, AttributeError):
        pass
    return None


def format_reward(completions, **kwargs) -> list[float]:
    """Reward for producing valid JSON with valid action fields.
    +1.0 for valid JSON with correct formation & play_type, -5.0 otherwise.
    """
    rewards = []
    for c in completions:
        action = _parse_action(_extract_text(c))
        rewards.append(1.0 if action else -5.0)
    return rewards


def play_reward(completions, defense_formation, **kwargs) -> list[float]:
    """Reward = sampled yards gained from real NFL data.
    Uses the lookup table: (formation, play_type, defense) -> sample outcome.
    Invalid/unparseable actions get -5.0.
    """
    rewards = []
    for c, defense in zip(completions, defense_formation):
        action = _parse_action(_extract_text(c))
        if action is None:
            rewards.append(-5.0)
            continue

        formation, play_type = action
        key = f"{formation}|{play_type}|{defense}"

        if key in LOOKUP:
            bucket = LOOKUP[key]
            idx = random.randrange(len(bucket["rewards"]))
            rewards.append(float(bucket["rewards"][idx]))
        else:
            # Fallback: pool all defenses for same (formation, play_type)
            prefix = f"{formation}|{play_type}|"
            fallback = [k for k in LOOKUP if k.startswith(prefix)]
            if fallback:
                all_r = []
                for fk in fallback:
                    all_r.extend(LOOKUP[fk]["rewards"])
                rewards.append(float(random.choice(all_r)))
            else:
                rewards.append(0.0)
    return rewards


# Quick test
test_valid = format_reward(['{"formation": "SHOTGUN", "play_type": "pass"}'])
test_invalid = format_reward(["not json"])
print(f"Valid format reward: {test_valid}  |  Invalid: {test_invalid}")

## Reward Logging Callback

Captures per-step rewards during training so we can plot improvement curves.

In [ ]:
# Cell 6: Logging callback

class RewardLogger(TrainerCallback):
    """Captures per-step reward metrics for plotting."""
    def __init__(self):
        self.steps = []
        self.rewards = []
        self.format_rewards = []
        self.play_rewards = []

    def on_log(self, args, state, control, logs=None, **kwargs):
        if not logs:
            return
        step = state.global_step
        if "reward" in logs:
            self.steps.append(step)
            self.rewards.append(logs["reward"])
        if "rewards/format_reward" in logs:
            self.format_rewards.append(logs["rewards/format_reward"])
        if "rewards/play_reward" in logs:
            self.play_rewards.append(logs["rewards/play_reward"])

reward_logger = RewardLogger()
print("RewardLogger ready")

## GRPO Training

GRPO generates 8 completions per prompt, ranks them within each group, and updates the policy to favor higher-reward completions. This is more sample-efficient than vanilla REINFORCE because the baseline is computed within each group rather than globally.

In [ ]:
# Cell 7: Configure and run GRPO training

OUTPUT_DIR = "checkpoints/football_grpo"
os.makedirs(OUTPUT_DIR, exist_ok=True)

training_args = GRPOConfig(
    output_dir=OUTPUT_DIR,
    # Optimizer
    learning_rate=5e-6,
    adam_beta1=0.9,
    adam_beta2=0.99,
    weight_decay=0.1,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    optim="paged_adamw_8bit",
    max_grad_norm=0.1,
    bf16=True,
    # Batching
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    # GRPO specifics
    num_generations=8,       # 8 completions per prompt, ranked within group
    num_iterations=1,
    beta=0.0,                # no KL penalty (aggressive exploration)
    # Generation
    max_prompt_length=256,
    max_completion_length=128,
    temperature=0.7,
    # Schedule
    max_steps=500,
    save_steps=100,
    logging_steps=1,
    report_to="none",
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[format_reward, play_reward],
    args=training_args,
    train_dataset=dataset,
    callbacks=[reward_logger],
)

print("Starting GRPO training...")
print(f"  {training_args.max_steps} steps, batch={training_args.per_device_train_batch_size}, "
      f"grad_accum={training_args.gradient_accumulation_steps}, num_generations={training_args.num_generations}")
trainer.train()

## Reward Curves

Plot training progress — we should see:
1. Format reward climbing toward +1.0 (model learns valid JSON)
2. Play reward (yards gained) improving as model discovers better play calls

In [ ]:
# Cell 8: Plot reward curves

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Total reward
ax = axes[0]
ax.plot(reward_logger.steps, reward_logger.rewards, "b-", alpha=0.3, label="Raw")
w = min(20, len(reward_logger.rewards))
if w > 1:
    ma = np.convolve(reward_logger.rewards, np.ones(w) / w, mode="valid")
    ax.plot(reward_logger.steps[w - 1:], ma, "r-", lw=2, label=f"MA-{w}")
ax.set_xlabel("Step")
ax.set_ylabel("Total Reward")
ax.set_title("GRPO Training: Total Reward")
ax.legend()
ax.grid(True, alpha=0.3)

# Component rewards
ax = axes[1]
if reward_logger.format_rewards:
    ax.plot(reward_logger.steps[:len(reward_logger.format_rewards)],
            reward_logger.format_rewards, "g-", alpha=0.5, label="Format (+1 valid)")
if reward_logger.play_rewards:
    ax.plot(reward_logger.steps[:len(reward_logger.play_rewards)],
            reward_logger.play_rewards, "b-", alpha=0.5, label="Play (yards)")
ax.set_xlabel("Step")
ax.set_ylabel("Reward")
ax.set_title("Reward Components")
ax.legend()
ax.grid(True, alpha=0.3)

fig.suptitle("NFL Play-Calling GRPO Training", fontsize=14, fontweight="bold")
fig.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "reward_curve.png"), dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved to {OUTPUT_DIR}/reward_curve.png")

## Save Model & Test

In [ ]:
# Cell 9: Save LoRA adapter
lora_path = os.path.join(OUTPUT_DIR, "lora_adapter")
model.save_lora(lora_path)
print(f"LoRA adapter saved to {lora_path}")

In [ ]:
# Cell 10: Test the trained model — run a few plays against different defenses

FastLanguageModel.for_inference(model)

test_defenses = ["Nickel (4-2-5)", "4-3", "Dime (3-2-6)", "3-4", "Nickel (3-3-5)"]

print("=== Trained Model Play Calls ===\n")
for defense in test_defenses:
    messages = make_prompt(defense)
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=64, temperature=0.3, do_sample=True)
    response = tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    action = _parse_action(response)
    status = "VALID" if action else "INVALID"
    print(f"Defense: {defense}")
    print(f"  Response: {response.strip()}")
    print(f"  Parsed: {action}  [{status}]")
    if action:
        formation, play_type = action
        key = f"{formation}|{play_type}|{defense}"
        if key in LOOKUP:
            avg_yards = np.mean(LOOKUP[key]["rewards"])
            print(f"  Avg yards (from data): {avg_yards:.1f}")
    print()